In [2]:
import pandas as pd
import numpy as np
from collections import Counter

In [3]:
#Question #1
pd.__version__

'3.0.2'

In [4]:
df=pd.read_csv('https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv')

In [5]:
#Question #2: How many records are in the dataset?
#Answer: there are 9704 records in the dataset
df.shape

(9704, 11)

In [6]:
df.head

<bound method NDFrame.head of       engine_displacement  num_cylinders  horsepower  vehicle_weight  \
0                     170            3.0       159.0     3413.433759   
1                     130            5.0        97.0     3149.664934   
2                     170            NaN        78.0     3079.038997   
3                     220            4.0         NaN     2542.392402   
4                     210            1.0       140.0     3460.870990   
...                   ...            ...         ...             ...   
9699                  140            5.0       164.0     2981.107371   
9700                  180            NaN       154.0     2439.525729   
9701                  220            2.0       138.0     2583.471318   
9702                  230            4.0       177.0     2905.527390   
9703                  270            3.0       140.0     2908.043477   

      acceleration  model_year  origin fuel_type         drivetrain  \
0             17.7        2003  Eu

In [7]:
#Question #3: How many fuel types are presented in the dataset?
#Answer: There are 2 fuel types
Counter(df['fuel_type'])

Counter({'Gasoline': 4898, 'Diesel': 4806})

In [8]:
#Question #4: How many columns in the dataset have missing values?
#Answer: There are 4 columns with missing values in the dataset
df.isnull().sum()

engine_displacement      0
num_cylinders          482
horsepower             708
vehicle_weight           0
acceleration           930
model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors              502
fuel_efficiency_mpg      0
dtype: int64

In [9]:
#Question #5: What's the maximum fuel efficiency of cars from Asia?
#Answer: 23.75

df.loc[
    df['origin'].str.lower() == 'asia',
    'fuel_efficiency_mpg'
].max()


np.float64(23.759122836520497)

In [10]:
#Question 6.1: Find the median value of the horsepower column in the dataset.
#Answer: 149.0

df.horsepower.median()

np.float64(149.0)

In [11]:
#Question 6.2: Next, calculate the most frequent value of the same horsepower column.
#Answer: 152.0
hp_mode = df.horsepower.mode()[0]
print(hp_mode)

152.0


In [12]:
#Question 6.3: Use the fillna method to fill the missing values in the horsepower column with the most frequent value from the previous step.

df['horsepower'] = df['horsepower'].fillna(hp_mode)
#checking to make sure it worked.
df.horsepower.isnull().sum() #there is no missing data now

np.int64(0)

In [13]:
#Question 6.4: Now, calculate the median value of horsepower once again. Has it changed?
#Answer: Yes, it has increased
df.horsepower.median()

np.float64(152.0)

In [14]:
#Question 7.1: Select all the cars from Asia

asia_cars=df[df['origin'] =='Asia']
#verifying that the selection worked:
asia_cars['origin'].value_counts() #all values in the origina column are "Asia", indicating that the code worked

origin
Asia    3247
Name: count, dtype: int64

In [16]:
#Question 7.2: Select only columns vehicle_weight and model_year
asia_cars=asia_cars[['vehicle_weight', 'model_year']]
#verifying that the selection worked
asia_cars.head


<bound method NDFrame.head of       vehicle_weight  model_year
8        2714.219310        2016
12       2783.868974        2010
14       3582.687368        2007
20       2231.808142        2011
21       2659.431451        2016
...              ...         ...
9688     3948.404625        2018
9692     3680.341381        2016
9693     2545.070139        2012
9698     3107.427820        2005
9703     2908.043477        2005

[3247 rows x 2 columns]>

In [19]:
#Question 7.3: Select the first 7 values
asia_cars=asia_cars.head(7)

#verifying that the selection worked
asia_cars.shape #the dataframe only has 7 rows, and 2 columns, which is what we expect

(7, 2)

In [21]:
#Question 7.4: Get the underlying NumPy array. Let's call it X.
X = asia_cars.to_numpy()
X.shape

(7, 2)

In [42]:
#Question 7.5: Compute matrix-matrix multiplication between the transpose of X and X. To get the transpose, use X.T. Let's call the result XTX

#writing function for vector-vector multiplication
def vector_vector_multiplication(u, v):
    assert u.shape[0] == v.shape[0]

    n=u.shape[0]

    result=0.0

    for i in range (n):
        result = result + u[i] * v[i]

    return result

#writing function for matrix-vector multiplication

def matrix_vector_multiplication(U, v):
    assert U.shape[1] == v.shape[0]

    num_rows = U.shape[0]
    result = np.zeros(num_rows)

    for i in range(num_rows):
        result[i] = vector_vector_multiplication(U[i], v)

    return result



#writing function for matrix-matrix multiplication

def matrix_matrix_multiplication(U, V):
    U=U.T #transposing matrix V
    assert U.shape[1] == V.shape[0]

    num_rows = U.shape[0]
    num_cols = V.shape[1]

    result = np.zeros((num_rows, num_cols))

    for i in range(num_cols):
        vi = V[:, i]                   
        Uvi = matrix_vector_multiplication(U, vi)
        result[:, i] = Uvi

    return result

#completing the multiplication
XTX=matrix_matrix_multiplication (X, X)
XTX

array([[62248334.33150762, 41431216.50732678],
       [41431216.50732678, 28373339.        ]])

In [41]:
#Question 7.6: Invert XTX

XTX_I= np.linalg.inv(XTX)
XTX_I

array([[ 5.71497081e-07, -8.34509443e-07],
       [-8.34509443e-07,  1.25380877e-06]])

In [44]:
#Question 7.7: Create an array y with values [1100, 1300, 800, 900, 1000, 1100, 1200].

y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])

In [47]:
#Question 7.8: Multiply the inverse of XTX with the transpose of X, and then multiply the result by y. Call the result w.

w = XTX_I @ X.T @ y
w

array([0.01386421, 0.5049067 ])

In [48]:
#Question 7.9: What's the sum of all the elements of the result?
#Answer: 0.52
w.sum()


np.float64(0.5187709081074003)